# MEDILENS — Exploratory Data Analysis

This notebook is for **exploration only**.  
Production logic lives in `src/`.  
Do not import from this notebook in `main.py`.

---

## Contents
1. Load raw data
2. Dataset overview
3. Missing value analysis
4. Target distribution
5. Numerical feature distributions
6. Categorical feature value counts
7. Correlation heatmap
8. Readmission rate by department

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib

matplotlib.rcParams['figure.dpi'] = 120

# Project root is one level up from notebooks/
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

from src.config import DATA_PATH, TARGET_COLUMN, NUMERICAL_COLS, CATEGORICAL_COLS

## 1. Load raw data

In [ ]:
df = pd.read_csv(DATA_PATH, parse_dates=['admission_date', 'discharge_date'])
print(f'Shape: {df.shape}')
df.head()

## 2. Dataset overview

In [ ]:
df.info()

In [ ]:
df.describe()

## 3. Missing value analysis

In [ ]:
missing = df.isnull().sum()
print(missing[missing > 0] if missing.any() else 'No missing values.')

## 4. Target distribution

In [ ]:
counts = df[TARGET_COLUMN].value_counts()
fig, ax = plt.subplots(figsize=(5, 3))
ax.bar(['Not Readmitted (0)', 'Readmitted (1)'], counts.values, color=['#4C9BE8', '#E85C5C'])
ax.set_title('Readmission Target Distribution')
ax.set_ylabel('Count')
for i, v in enumerate(counts.values):
    ax.text(i, v + 5, str(v), ha='center')
plt.tight_layout()
plt.show()
print(f'Readmission rate: {df[TARGET_COLUMN].mean()*100:.1f}%')

## 5. Numerical feature distributions

In [ ]:
fig, axes = plt.subplots(1, len(NUMERICAL_COLS), figsize=(18, 3))
for ax, col in zip(axes, NUMERICAL_COLS):
    df[col].hist(ax=ax, bins=20, color='#4C9BE8', edgecolor='white')
    ax.set_title(col)
plt.suptitle('Numerical Feature Distributions', y=1.02)
plt.tight_layout()
plt.show()

## 6. Categorical feature value counts

In [ ]:
for col in CATEGORICAL_COLS:
    print(f'\n{col}:')
    print(df[col].value_counts().to_string())

## 7. Correlation heatmap (numerical features)

In [ ]:
corr = df[NUMERICAL_COLS + [TARGET_COLUMN]].corr()
fig, ax = plt.subplots(figsize=(7, 5))
im = ax.imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
ax.set_xticks(range(len(corr.columns)))
ax.set_yticks(range(len(corr.columns)))
ax.set_xticklabels(corr.columns, rotation=45, ha='right')
ax.set_yticklabels(corr.columns)
plt.colorbar(im, ax=ax)
ax.set_title('Correlation Matrix')
plt.tight_layout()
plt.show()

## 8. Readmission rate by department

In [ ]:
dept_rate = df.groupby('department')[TARGET_COLUMN].mean().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(7, 3))
dept_rate.plot(kind='bar', color='#E85C5C', ax=ax)
ax.set_ylabel('Readmission Rate')
ax.set_title('30-Day Readmission Rate by Department')
ax.set_xticklabels(dept_rate.index, rotation=30, ha='right')
plt.tight_layout()
plt.show()